In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

disable_all_ps()


# ----------------------------------------------------------------------------
print("██████  ██ ███████  ██████  ██████  ███    ██ ███    ██ ███████  ██████ ████████     ██       ██████   █████  ██████  ")
print("██   ██ ██ ██      ██      ██    ██ ████   ██ ████   ██ ██      ██         ██        ██      ██    ██ ██   ██ ██   ██ ")
print("██   ██ ██ ███████ ██      ██    ██ ██ ██  ██ ██ ██  ██ █████   ██         ██        ██      ██    ██ ███████ ██   ██ ")
print("██   ██ ██      ██ ██      ██    ██ ██  ██ ██ ██  ██ ██ ██      ██         ██        ██      ██    ██ ██   ██ ██   ██ ")
print("██████  ██ ███████  ██████  ██████  ██   ████ ██   ████ ███████  ██████    ██        ███████  ██████  ██   ██ ██████ ")
# ----------------------------------------------------------------------------

# Test A11 : V_INFO test for VIN and VOUT

kt : ---> VDD3V3

ks : ---> dm.ch1 (IOUT)

ps.ch1 : ---> relay power

ps.ch2 : ---> relay.ch3 (EN)

dm.ch1 ---> relay.ch1 (VIN)

dm.ch1 +--> relay.ch2 (VOUT)

dm.ch2 : ---> VOUT and GND

dm.ch3 : ---> V_INFO and GND

dm.ch4 : ---> VIN and GND

relay.ch1 : VIN

relay.ch2 : VOUT

relay.ch3 : EN

eloader : disconnect

In [ ]:
temperature = "N40C"

log.output_set_filename(f"[{temperature}_11] V_INFO")
log.output_csv(["VIN (V)", "VDD3V3 (V)", "EN (V)", "V_INFO (V)", "RATIO (V_INFO/VIN)", "CONV_VIN_EN"])

In [ ]:
disable_all_ps()
relay.init_channels

v_vdd = 3.3
v_en = 0

v_start  = 1
v_finish = 30.01
step = 1
ndigit = 0

list_temp = list(np.arange(v_start, v_finish, step))
list_vin = [round(num, ndigit) for num in list_temp]

print(f"voltage step : {step}, round : {ndigit}")

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

# --------------------------------------------
ks.cfg_all = list_vin[0], 0.5 # vin
ks.enable
delay(0.5)

dm.ch1.current_200mA
relay.ch1.enable # vin
relay.ch2.enable
delay(0.5)

# --------------------------------------------
ps.ch2.cfg_all = v_en, 0.1
ps.ch2.enable

delay(0.5)
relay.ch3.enable # en
delay(0.5)
# --------------------------------------------

ic.CONV_VIN_EN = 1

In [ ]:
# ----------------------------------------------------------------------------

for vin in list_vin:
    
    ks.vset = vin
    
    m_vin  = round(dm.ch4.voltage_200V, 6)
    v_info = round(dm.ch3.voltage_2V, 6) # V scale
    reg_conv = ic.CONV_VIN_EN
    ratio = round(v_info / m_vin, 6)
    
    ret = [m_vin, v_vdd, v_en, v_info, ratio, reg_conv]
    log.output_csv(ret)
    
    print(ret)

# ----------------------------------------------------------------------------

ks.vset = list_vin[0]
delay(1)

v_en = 3.3
ps.ch2.vset = v_en
delay(1)

log.output_csv(["VIN (V)", "VDD3V3 (V)", "EN (V)", "V_INFO (V)", "RATIO (V_INFO/VIN)", "CONV_VIN_EN"])

for vin in list_vin:
    
    ks.vset = vin
    
    m_vin  = round(dm.ch4.voltage_200V, 6)
    v_info = round(dm.ch3.voltage_2V, 6) # V scale
    reg_conv = ic.CONV_VIN_EN
    ratio = round(v_info / m_vin, 6)
    
    ret = [m_vin, v_vdd, v_en, v_info, ratio, reg_conv]
    log.output_csv(ret)
    
    print(ret)

# ----------------------------------------------------------------------------

ks.vset = list_vin[0]
delay(1)

v_en = 0
ps.ch2.vset = v_en
delay(1)

relay.ch1.disable
relay.ch2.enable

ic.CONV_VIN_EN = 0
ic.CONV_VOUT_EN = 1

log.output_csv(["VOUT (V)", "VDD3V3 (V)", "EN (V)", "V_INFO (V)", "RATIO (V_INFO/VIN)", "CONV_VOUT_EN"])

for vin in list_vin:
    
    ks.vset = vin
    
    m_vout = round(dm.ch2.voltage_200V, 6)
    v_info = round(dm.ch3.voltage_2V, 6) # V scale
    reg_conv = ic.CONV_VOUT_EN
    ratio = round(v_info / m_vout, 6)
    
    ret = [m_vout, v_vdd, v_en, v_info, ratio, reg_conv]
    log.output_csv(ret)
    
    print(ret)

# ----------------------------------------------------------------------------

ks.vset = list_vin[0]
delay(1)

v_en = 3.3
ps.ch2.vset = v_en
delay(1)

log.output_csv(["VOUT (V)", "VDD3V3 (V)", "EN (V)", "V_INFO (V)", "RATIO (V_INFO/VIN)", "CONV_VOUT_EN"])

for vin in list_vin:
    
    ks.vset = vin
    
    m_vout = round(dm.ch2.voltage_200V, 6)
    v_info = round(dm.ch3.voltage_2V, 6) # V scale
    reg_conv = ic.CONV_VOUT_EN
    ratio = round(v_info / m_vout, 6)
    
    ret = [m_vout, v_vdd, v_en, v_info, ratio, reg_conv]
    log.output_csv(ret)
    
    print(ret)

# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()